In [24]:
import sys
import ollama

print(f"Active Python Executable: {sys.executable}")

client = ollama.Client(host="http://host.docker.internal:11434")
print("Installed Models:", [m.model for m in client.list().models])

Active Python Executable: /workspace/.venv/bin/python
Installed Models: ['mistral:latest', 'nomic-embed-text:latest', 'qwen2.5-coder:1.5b-base', 'llama3.1:8b', 'qwen2.5vl:7b', 'llama3.2-vision:latest']


In [6]:
import ollama

client = ollama.Client(host="http://host.docker.internal:11434")

# List active models currently residing in RAM
running = client.ps()
print(running)

models=[Model(model='llama3.1:8b', name='llama3.1:8b', digest='46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e', expires_at=datetime.datetime(2026, 10, 20, 12, 46, 5, 518008, tzinfo=TzInfo(3600)), size=5584516545, size_vram=0, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='8.0B', quantization_level='Q4_K_M'), context_length=4096)]


Run Models and save data to CSV

In [34]:
from datetime import datetime
import os
import time
import ollama
import pandas as pd

# 1. Connect to host Ollama instance
client = ollama.Client(host="http://host.docker.internal:11434")

MODELS = ["llama3.1:8b", "mistral:latest"]
PROMPT = "Write a 3-sentence summary of why Docker is useful in software engineering."
NUM_LOOPS = 3

# File management settings
CSV_DIR = "outputs"
CSV_PATH = os.path.join(CSV_DIR, "benchmarks.csv")
MAX_CSV_MB = 5.0  # Max size in MB before auto-archiving

results = []


def unload_model(model_name: str):
    """Forces Ollama to drop the model from RAM immediately."""
    try:
        client.chat(model=model_name, messages=[], keep_alive=0)
    except Exception:
        pass


def manage_csv_size(file_path: str, max_mb: float):
    """Archives the CSV if it exceeds max_mb size."""
    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        if size_mb >= max_mb:
            archive_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            archive_path = os.path.join(
                os.path.dirname(file_path),
                f"benchmarks_archive_{archive_timestamp}.csv",
            )
            os.rename(file_path, archive_path)
            print(
                f"[Storage Notice] CSV reached {size_mb:.2f} MB. Archived to: {archive_path}"
            )


def save_benchmark_csv(df: pd.DataFrame, file_path: str, max_mb: float):
    """Appends benchmark results to CSV, creating directories and handling file rolling as needed."""
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    manage_csv_size(file_path, max_mb)

    # Append header only if file does not exist
    file_exists = os.path.exists(file_path)
    df.to_csv(file_path, mode="a", index=False, header=not file_exists)
    print(f"[Storage] Benchmarks appended to {file_path}")


def run_model(model_name: str, prompt: str):
    print("=" * 80)
    print(f" Benchmarking Model: {model_name}")
    print("=" * 80 + "\n")

    for i in range(1, NUM_LOOPS + 1):
        run_time_iso = datetime.now().isoformat()
        print(f"--- Iteration {i}/{NUM_LOOPS} [{run_time_iso}] ---")

        # Step A: Force unload model from RAM for a true Cold Start
        unload_model(model_name)
        time.sleep(1)  # Brief pause for OS memory deallocation

        # Step B: COLD RUN
        start_cold = time.perf_counter()
        cold_response = client.chat(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            keep_alive="5m",
        )
        cold_time = time.perf_counter() - start_cold

        cold_eval_count = cold_response.get("eval_count", 0)
        cold_eval_duration_sec = cold_response.get("eval_duration", 0) / 1e9
        cold_result = cold_response["message"]["content"]

        print(
            f"  [Cold Start] Time: {cold_time:.2f}s | Output Tokens: {cold_eval_count}"
        )

        # Step C: WARM RUN
        start_warm = time.perf_counter()
        warm_response = client.chat(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            keep_alive="5m",
        )
        warm_time = time.perf_counter() - start_warm

        warm_eval_count = warm_response.get("eval_count", 0)
        warm_eval_duration_sec = warm_response.get("eval_duration", 0) / 1e9
        warm_result = warm_response["message"]["content"]

        print(
            f"  [Warm Run]   Time: {warm_time:.2f}s | Output Tokens: {warm_eval_count}"
        )

        load_overhead = cold_time - warm_time
        print(f"  -> Model Loading Overhead: ~{load_overhead:.2f}s\n")

        # 1. PRINT FULL MULTI-LINE TEXT IN CONSOLE FOR QUALITY COMPARISON
        print("┌" + "─" * 78)
        print(f"│ COLD OUTPUT ({model_name}):")
        print(f"│ {cold_result.strip()}")
        print("├" + "─" * 78)
        print(f"│ WARM OUTPUT ({model_name}):")
        print(f"│ {warm_result.strip()}")
        print("└" + "─" * 78 + "\n")

        # 2. LOG METRICS (EXCLUDING TEXT RESULT FROM DATAFRAME)
        results.append({
            "Timestamp": run_time_iso,
            "Model_Name": model_name,
            "Iteration": i,
            "Cold_Time_Sec": round(cold_time, 3),
            "Warm_Time_Sec": round(warm_time, 3),
            "Load_Overhead_Sec": round(load_overhead, 3),
            "Cold_Tok_Per_Sec": round(
                cold_eval_count / cold_eval_duration_sec, 2
            )
            if cold_eval_duration_sec
            else 0,
            "Warm_Tok_Per_Sec": round(
                warm_eval_count / warm_eval_duration_sec, 2
            )
            if warm_eval_duration_sec
            else 0,
            "Cold_Output": cold_result,
            "Warm_Output": warm_result
        })

    # Cleanup: Unload model from RAM
    unload_model(model_name)


# Execute benchmarking across all models
print(
    f"Starting Cold vs. Warm Benchmarking for {MODELS} over {NUM_LOOPS} iterations...\n"
)
for model_name in MODELS:
    run_model(model_name, PROMPT)
    print(f"Completed benchmarking for model: {model_name}\n")

# Display numerical summary table for current run
df_results = pd.DataFrame(results)
display(df_results)

# Save/Append to CSV with file size management
save_benchmark_csv(df_results, CSV_PATH, MAX_CSV_MB)

Starting Cold vs. Warm Benchmarking for ['llama3.1:8b', 'mistral:latest'] over 3 iterations...

 Benchmarking Model: llama3.1:8b

--- Iteration 1/3 [2026-10-20T13:37:29.173134] ---
  [Cold Start] Time: 15.85s | Output Tokens: 113
  [Warm Run]   Time: 12.18s | Output Tokens: 111
  -> Model Loading Overhead: ~3.67s

┌──────────────────────────────────────────────────────────────────────────────
│ COLD OUTPUT (llama3.1:8b):
│ Here is a 3-sentence summary of the usefulness of Docker in software engineering:

Docker is a powerful tool in software engineering because it allows developers to package their applications and dependencies into a single container that can be easily shared, replicated, and deployed across different environments. This eliminates the "it works on my machine" problem by ensuring that the application behaves consistently and reliably in any environment, from development to production. By providing a consistent and portable way to deploy applications, Docker helps teams

,Timestamp,Model_Name,Iteration,Cold_Time_Sec,Warm_Time_Sec,Load_Overhead_Sec,Cold_Tok_Per_Sec,Warm_Tok_Per_Sec,Cold_Output,Warm_Output
0,2026-10-20T13:37:29.173134,llama3.1:8b,1,15.853,12.184,3.669,9.67,8.75,Here is a 3-sentence summary of the usefulness...,Here is a 3-sentence summary of why Docker is ...
1,2026-10-20T13:37:58.933514,llama3.1:8b,2,18.734,17.891,0.842,7.49,6.12,Here is a 3-sentence summary of why Docker is ...,Here is a 3-sentence summary of why Docker is ...
2,2026-10-20T13:38:39.592602,llama3.1:8b,3,20.037,18.723,1.315,6.52,5.81,Here is a 3-sentence summary of why Docker is ...,Here is a 3-sentence summary of why Docker is ...
3,2026-10-20T13:39:20.837540,mistral:latest,1,21.976,14.477,7.499,6.78,5.86,Docker is useful in software engineering for ...,Docker is beneficial in software engineering ...
4,2026-10-20T13:39:59.981525,mistral:latest,2,20.907,16.521,4.386,6.20,6.23,Docker is useful in software engineering for ...,Docker is useful in software engineering for ...
5,2026-10-20T13:40:39.908899,mistral:latest,3,12.117,13.249,-1.132,6.35,6.20,Docker is useful in software engineering due ...,Docker is beneficial in software engineering ...


[Storage] Benchmarks appended to outputs/benchmarks.csv


Compare Model Performance and get recommendation

In [2]:
import os
import pandas as pd
import ollama
import re
import numpy as np
from collections.abc import Hashable

# Define paths matching your project directory
CSV_DIR = "outputs"
CSV_PATH = os.path.join(CSV_DIR, "benchmarks.csv")

# Configuration for Local LLM Reviewer & Embedding Model
REVIEWER_MODEL = "mistral:latest"
EMBEDDING_MODEL = "nomic-embed-text:latest"


def evaluate_output_structure(text: str) -> dict:
    """Evaluates structural constraints of a generated LLM text output."""
    if not isinstance(text, str):
        text = ""

    # Split text into sentences based on period termination
    sentences = [s for s in text.split(".") if s.strip()]

    return {
        "sentence_count": len(sentences),
        "meets_3_sentence_constraint": len(sentences) == 3,
        "char_length": len(text),
        "mentions_docker": "docker" in text.lower(),
    }


def llm_rate_output(text: str, row_idx: Hashable, run_type: str) -> float:
    """Forces the local review model to return a score from 1 to 5, using robust regex extraction."""
    if not isinstance(text, str) or not text.strip():
        print(f"  [{run_type} Row {row_idx}] Empty text -> Defaulting to 0.0")
        return 0.0

    prompt = f"""Task: Rate the following response from 1 to 5 based on how well it explains why Docker is useful in software engineering and whether it meets a concise 3-sentence constraint.
Format requirement: Your response must start with a single digit score (1, 2, 3, 4, or 5).

Response to evaluate:
"{text}"

Score (1-5):"""

    try:
        client = ollama.Client(host="http://host.docker.internal:11434")
        response = client.chat(
            model=REVIEWER_MODEL,
            messages=[{"role": "user", "content": prompt}],
            options={
                "temperature": 0.0,
                "num_predict": 5  # Allow a few tokens to complete prefix safely
            },
        )
        score_text = response["message"]["content"].strip()
        print(f"  [{run_type} Row {row_idx}] Raw Judge Output: {repr(score_text)}")

        # Use regular expression to find the first standalone digit from 1 to 5
        match = re.search(r'\b([1-5])\b', score_text)
        if match:
            return float(match.group(1))

        # Fallback: search for any digit 1-5 in the text if word boundaries fail
        for char in score_text:
            if char.isdigit():
                val = float(char)
                if 1.0 <= val <= 5.0:
                    return val

        print(f"  ⚠️ [{run_type} Row {row_idx}] No valid rating found in text, using 0.0")
        return 0.0
    except Exception as e:
        print(f"  ❌ [{run_type} Row {row_idx}] Reviewer Error: {e}")
        return 0.0


def compute_cosine_similarity(text1: str, text2: str) -> float:
    """Computes cosine similarity between two texts using local nomic-embed-text embeddings."""
    if not isinstance(text1, str) or not text1.strip() or not isinstance(text2, str) or not text2.strip():
        return 0.0

    try:
        client = ollama.Client(host="http://host.docker.internal:11434")

        res1 = client.embeddings(model=EMBEDDING_MODEL, prompt=text1)
        res2 = client.embeddings(model=EMBEDDING_MODEL, prompt=text2)

        vec1 = np.array(res1["embedding"])
        vec2 = np.array(res2["embedding"])

        # Calculate cosine similarity
        dot_product = np.dot(vec1, vec2)
        norm1 = np.linalg.norm(vec1)
        norm2 = np.linalg.norm(vec2)

        if norm1 == 0 or norm2 == 0:
            return 0.0

        return float(dot_product / (norm1 * norm2))
    except Exception as e:
        print(f"  ❌ Embedding Error: {e}")
        return 0.0


def llm_select_optimal_model(summary_text: str, failure_report_text: str) -> str:
    """Asks the local reviewer model to evaluate metrics and select the optimal model cleanly without looping."""
    prompt = f"""You are an expert AI Benchmark Engineer.
Review the aggregated metrics and failure reports below and select the single optimal model between 'llama3.1:8b' and 'mistral:latest'.

Selection Guidelines:
- The winner is the model that unconditionally passes the structural failure audit while maximizing the LLM-as-a-Judge Quality Rating and maintaining strong throughput.
- If a model fails the 3-sentence constraint even once, it is automatically disqualified for production use cases with strict output formatting rules.

Aggregated Metrics Summary:
{summary_text}

Structural Failure Audit Report:
{failure_report_text}

Provide your final optimal model selection and a short 2-3 sentence rationale based strictly on the data above:"""

    try:
        client = ollama.Client(host="http://host.docker.internal:11434")
        response = client.chat(
            model=REVIEWER_MODEL,
            messages=[{"role": "user", "content": prompt}],
            options={
                "temperature": 0.0,
                "num_predict": 150  # Restrict token generation length to prevent infinite loops
            },
        )
        return response["message"]["content"].strip()
    except Exception as e:
        return f"Error running reviewer model selection: {e}"


def analyze_model_outputs_and_metrics(csv_file_path: str):
    """Loads benchmarks CSV, evaluates output structure, scores text quality (1-5),
    computes cold-vs-warm embedding cosine similarity, reports metrics grouped by model,
    and delegates final model selection to the local reviewer model.
    """
    if not os.path.exists(csv_file_path):
        print(f"[Error] Benchmark file not found at: {csv_file_path}")
        return None

    # Load dataset
    df = pd.read_csv(csv_file_path)
    print(f"Loaded {len(df)} total run records from {csv_file_path}\n")

    # Warning check if reviewing with a model present in the generation dataset
    unique_gen_models = df["Model_Name"].unique()
    if REVIEWER_MODEL in unique_gen_models:
        print(
            f"⚠️ [Warning] Reviewer model '{REVIEWER_MODEL}' is also present in the generation dataset. (Proceeding with evaluation).\n"
        )

    # 1. Run LLM-as-a-Judge ratings & Semantic Similarity Checks
    if "Cold_Output" in df.columns and "Warm_Output" in df.columns:
        print(
            f"Running LLM-as-a-Judge evaluation using '{REVIEWER_MODEL}' & Semantic Drift Check using '{EMBEDDING_MODEL}'...\n"
        )
        cold_ratings = []
        warm_ratings = []
        semantic_similarities = []

        for idx, row in df.iterrows():
            print(
                f"Evaluating Iteration {row.get('Iteration', idx)} for {row.get('Model_Name', 'Unknown')}..."
            )
            cold_text = row.get("Cold_Output", "")
            warm_text = row.get("Warm_Output", "")

            cold_ratings.append(llm_rate_output(cold_text, idx, "Cold"))
            warm_ratings.append(llm_rate_output(warm_text, idx, "Warm"))

            # Compute semantic consistency between cold and warm outputs
            sim_score = compute_cosine_similarity(cold_text, warm_text)
            semantic_similarities.append(sim_score)
            print(f"  [Semantic Check] Cold vs Warm Cosine Similarity: {sim_score:.4f}")

        df["Cold_Quality_Rating"] = cold_ratings
        df["Warm_Quality_Rating"] = warm_ratings
        df["Cold_Warm_Cosine_Similarity"] = semantic_similarities
        print("\n")

    # 2. GROUPED PERFORMANCE, THROUGHPUT, RATINGS & SEMANTIC DRIFT
    group_cols = [
        "Cold_Time_Sec",
        "Warm_Time_Sec",
        "Cold_Tok_Per_Sec",
        "Warm_Tok_Per_Sec",
    ]
    if "Cold_Quality_Rating" in df.columns:
        group_cols.extend(["Cold_Quality_Rating", "Warm_Quality_Rating"])
    if "Cold_Warm_Cosine_Similarity" in df.columns:
        group_cols.append("Cold_Warm_Cosine_Similarity")

    metrics_summary = df.groupby("Model_Name")[group_cols].mean().round(2)

    print("=" * 85)
    print(" AGGREGATED METRICS, RATINGS & SEMANTIC DRIFT (GROUPED BY MODEL)")
    print("=" * 85)
    display(metrics_summary)
    print("\n")

    # 3. STRUCTURAL EVALUATION & FAILURE REPORTING
    print("=" * 85)
    print(" STRUCTURAL EVALUATION & FAILURE AUDIT")
    print("=" * 85)

    failures = []
    for idx, row in df.iterrows():
        for run_type, col_name in [
            ("Cold", "Cold_Output"),
            ("Warm", "Warm_Output"),
        ]:
            text_val = row.get(col_name, "")
            metrics = evaluate_output_structure(text_val)

            failed_constraints = (
                not metrics["meets_3_sentence_constraint"]
                or not metrics["mentions_docker"]
            )

            if failed_constraints:
                failures.append({
                    "Model_Name": row["Model_Name"],
                    "Run_Type": run_type,
                    "Iteration": row["Iteration"],
                    "Sentence_Count": metrics["sentence_count"],
                    "Mentions_Docker": metrics["mentions_docker"],
                })

    failure_report_text = "No structural failures detected."
    if failures:
        print(f"⚠️  [Alert] Found {len(failures)} failed generation(s):\n")
        df_failures = pd.DataFrame(failures)
        display(df_failures)
        failure_report_text = df_failures.to_string()
    else:
        print(
            "✅ All model outputs successfully met structural constraints (3 sentences & mentioned 'Docker')!"
        )

    # 4. DELEGATE OPTIMAL MODEL SELECTION TO REVIEWER MODEL
    print("\n" + "=" * 85)
    print(f" LLM-AS-A-JUDGE: OPTIMAL MODEL SELECTION ({REVIEWER_MODEL})")
    print("=" * 85)

    summary_text = metrics_summary.to_string()
    selection_evaluation = llm_select_optimal_model(summary_text, failure_report_text)
    print(selection_evaluation)
    print("=" * 85)


# Execute evaluation
analyze_model_outputs_and_metrics(CSV_PATH)

Loaded 6 total run records from outputs/benchmarks.csv

⚠️ [Warning] Reviewer model 'mistral:latest' is also present in the generation dataset. (Proceeding with evaluation).

Running LLM-as-a-Judge evaluation using 'mistral:latest' & Semantic Drift Check using 'nomic-embed-text:latest'...

Evaluating Iteration 1 for llama3.1:8b...
  [Cold Row 0] Raw Judge Output: '5\n\nThe'
  [Warm Row 0] Raw Judge Output: '5\n\nThe'
  [Semantic Check] Cold vs Warm Cosine Similarity: 0.9837
Evaluating Iteration 2 for llama3.1:8b...
  [Cold Row 1] Raw Judge Output: '5\n\nThe'
  [Warm Row 1] Raw Judge Output: '5\n\nThe'
  [Semantic Check] Cold vs Warm Cosine Similarity: 0.9921
Evaluating Iteration 3 for llama3.1:8b...
  [Cold Row 2] Raw Judge Output: '5\n\nThe'
  [Warm Row 2] Raw Judge Output: '5\n\nThe'
  [Semantic Check] Cold vs Warm Cosine Similarity: 0.9881
Evaluating Iteration 1 for mistral:latest...
  [Cold Row 3] Raw Judge Output: '5\n\nThe'
  [Warm Row 3] Raw Judge Output: '5\n\nThe'
  [Semantic 

,Cold_Time_Sec,Warm_Time_Sec,Cold_Tok_Per_Sec,Warm_Tok_Per_Sec,Cold_Quality_Rating,Warm_Quality_Rating,Cold_Warm_Cosine_Similarity
Model_Name,,,,,,,
llama3.1:8b,18.21,16.27,7.89,6.89,5.0,5.0,0.99
mistral:latest,18.33,14.75,6.44,6.10,5.0,5.0,0.98




 STRUCTURAL EVALUATION & FAILURE AUDIT
⚠️  [Alert] Found 5 failed generation(s):



,Model_Name,Run_Type,Iteration,Sentence_Count,Mentions_Docker
0,mistral:latest,Warm,1,4,True
1,mistral:latest,Cold,2,4,True
2,mistral:latest,Warm,2,4,True
3,mistral:latest,Cold,3,1,True
4,mistral:latest,Warm,3,1,True



 LLM-AS-A-JUDGE: OPTIMAL MODEL SELECTION (mistral:latest)
Based on the provided data, the optimal model for the given criteria would be 'llama3.1:8b'. The reason for this selection is that 'llama3.1:8b' consistently outperforms 'mistral:latest' in terms of throughput, with higher Cold_Tok_Per_Sec and Warm_Tok_Per_Sec rates. Additionally, 'llama3.1:8b' does not have any failures in the structural failure audit, while 'mistral:latest' fails the audit in all reported runs. Therefore, 'llama3.1:8b' is the more suitable model for production use cases with strict output formatting rules and a need for strong
